# 🎯 SQL: Window Functions — The Complete Guide

**Advanced SQL Concepts for Interview Preparation**

- **Created:** 2026-02-28
- **Focus:** `ROW_NUMBER`, `RANK`, `DENSE_RANK`, `LAG`/`LEAD`, Rolling Aggregates
- **Tags:** `sql` | `window-functions` | `analytics` | `interview-prep`
- **Runtime:** DuckDB (in-process, zero setup)

In [ ]:
# Setup: DuckDB — zero config, runs in-process, full SQL
# pip install duckdb
import duckdb

# Create an in-memory connection
con = duckdb.connect()

# -------------------------------------------------------
# Seed data: Citi-style server telemetry (employees for ranking demos)
# -------------------------------------------------------
con.execute("""
CREATE TABLE employees AS
SELECT * FROM (VALUES
    ('Alice',   'Engineering', 120000),
    ('Bob',     'Engineering', 120000),
    ('Charlie', 'Engineering', 110000),
    ('Diana',   'Engineering',  95000),
    ('Eve',     'Marketing',   105000),
    ('Frank',   'Marketing',   105000),
    ('Grace',   'Marketing',    88000),
    ('Henry',   'Finance',     115000),
    ('Ivy',     'Finance',     115000),
    ('Jack',    'Finance',      92000)
) t(name, department, salary)
""")

con.execute("""
CREATE TABLE server_metrics AS
SELECT * FROM (VALUES
    ('server-01', DATE '2026-01-01', 55.0),
    ('server-01', DATE '2026-01-02', 62.0),
    ('server-01', DATE '2026-01-03', 58.0),
    ('server-01', DATE '2026-01-04', 71.0),
    ('server-01', DATE '2026-01-05', 89.0),
    ('server-01', DATE '2026-01-06', 94.0),
    ('server-01', DATE '2026-01-07', 88.0),
    ('server-02', DATE '2026-01-01', 40.0),
    ('server-02', DATE '2026-01-02', 45.0),
    ('server-02', DATE '2026-01-03', 43.0),
    ('server-02', DATE '2026-01-04', 50.0),
    ('server-02', DATE '2026-01-05', 48.0),
    ('server-02', DATE '2026-01-06', 52.0),
    ('server-02', DATE '2026-01-07', 55.0)
) t(server_id, collection_date, cpu_utilization)
""")

print("✅ Tables created: employees, server_metrics")
con.execute("SELECT * FROM employees").df()

## 📖 The Core Concept in Plain Language

A **window function** performs a calculation across a set of rows **related to the current row** — without collapsing the rows like `GROUP BY` does.

| Approach | What it does | Row count change |
|----------|-------------|------------------|
| `GROUP BY dept` | One row **per department** | **Collapses** rows |
| `OVER (PARTITION BY dept)` | Calculation **alongside** each row | **Keeps all rows** |

### The Syntax Template

```sql
window_function() OVER (
    PARTITION BY column1        -- defines the "window" (group)
    ORDER BY column2            -- defines ordering within the window
    ROWS BETWEEN n PRECEDING    -- optional: defines the frame
         AND CURRENT ROW
)
```

### The Concise Cheat Code

| Function | Nickname | Behavior on Ties |
|----------|----------|-------------------|
| `ROW_NUMBER()` | The Stubborn Counter | Assigns unique numbers — one person gets 1, the other gets 2 |
| `RANK()` | The Olympic Medal System | Ties share rank; **skips** numbers (1, 1, **3**) |
| `DENSE_RANK()` | The Fair System | Ties share rank; **never skips** (1, 1, **2**) — use for Top-N |
| `LAG(col, n)` | Look Backwards | Returns the value `n` rows *before* the current row |
| `LEAD(col, n)` | Look Forwards | Returns the value `n` rows *after* the current row |
| `SUM()` / `AVG()` over frame | Rolling Math | Aggregate over a sliding window of rows |

### ⚠️ The Classic Interview Trap

You **CANNOT** put a window function in a `WHERE` clause. Window functions are evaluated *after* `WHERE`. Wrap in a **CTE first**, then filter:

```sql
-- ❌ WRONG:
SELECT * FROM employees
WHERE DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) <= 3

-- ✅ CORRECT:
WITH ranked AS (
    SELECT *, DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS rnk
    FROM employees
)
SELECT * FROM ranked WHERE rnk <= 3
```

## 🏆 Part 1: The Rankers — ROW_NUMBER, RANK, DENSE_RANK

In [ ]:
# Side-by-side comparison: all three ranking functions on the same data
# Notice the tie-handling difference for Alice & Bob (both earn 120,000)

result = con.execute("""
SELECT
    name,
    department,
    salary,
    ROW_NUMBER()  OVER (PARTITION BY department ORDER BY salary DESC) AS row_num,
    RANK()        OVER (PARTITION BY department ORDER BY salary DESC) AS rank,
    DENSE_RANK()  OVER (PARTITION BY department ORDER BY salary DESC) AS dense_rank
FROM employees
ORDER BY department, salary DESC
""").df()

print(result.to_string(index=False))

In [ ]:
# Real interview task: Top 2 earners per department
# DENSE_RANK is correct here — RANK would skip the 3rd person unfairly

result = con.execute("""
WITH ranked AS (
    SELECT
        name,
        department,
        salary,
        DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS rnk
    FROM employees
)
SELECT name, department, salary, rnk
FROM ranked
WHERE rnk <= 2
ORDER BY department, rnk
""").df()

print("Top 2 earners per department:")
print(result.to_string(index=False))

## ⏱️ Part 2: Time Travel — LAG() and LEAD()

In [ ]:
# Real Citi use case: detect CPU utilization spikes
# Compare each day's utilization to the previous day — no self-join needed

result = con.execute("""
SELECT
    server_id,
    collection_date,
    cpu_utilization,
    LAG(cpu_utilization, 1)  OVER (PARTITION BY server_id ORDER BY collection_date)
        AS prev_day_cpu,
    LEAD(cpu_utilization, 1) OVER (PARTITION BY server_id ORDER BY collection_date)
        AS next_day_cpu,
    ROUND(
        cpu_utilization
        - LAG(cpu_utilization, 1) OVER (PARTITION BY server_id ORDER BY collection_date),
    1) AS day_over_day_delta
FROM server_metrics
ORDER BY server_id, collection_date
""").df()

print(result.to_string(index=False))

In [ ]:
# Alert: servers where CPU jumped more than 15% day-over-day
# This is the exact pattern used in Citi APM spike detection

result = con.execute("""
WITH daily_deltas AS (
    SELECT
        server_id,
        collection_date,
        cpu_utilization,
        LAG(cpu_utilization, 1) OVER (PARTITION BY server_id ORDER BY collection_date) AS prev_cpu,
        cpu_utilization
        - LAG(cpu_utilization, 1) OVER (PARTITION BY server_id ORDER BY collection_date) AS delta
    FROM server_metrics
)
SELECT server_id, collection_date, prev_cpu, cpu_utilization, ROUND(delta, 1) AS spike
FROM daily_deltas
WHERE delta > 15
ORDER BY spike DESC
""").df()

print("⚠️  Servers with CPU spike > 15 points:")
print(result.to_string(index=False))

## 📊 Part 3: Rolling Aggregates — Running Totals and Moving Averages

In [ ]:
# Rolling 3-day average CPU utilization per server
# ROWS BETWEEN: physical rows (predictable)
# RANGE BETWEEN: logical rows with same ORDER BY value (avoid for time series)

result = con.execute("""
SELECT
    server_id,
    collection_date,
    cpu_utilization,
    ROUND(AVG(cpu_utilization) OVER (
        PARTITION BY server_id
        ORDER BY collection_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW   -- 3-day window
    ), 1) AS rolling_3day_avg,
    ROUND(AVG(cpu_utilization) OVER (
        PARTITION BY server_id
        ORDER BY collection_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW   -- 7-day window
    ), 1) AS rolling_7day_avg,
    SUM(cpu_utilization) OVER (
        PARTITION BY server_id
        ORDER BY collection_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW  -- running total
    ) AS running_total_cpu
FROM server_metrics
ORDER BY server_id, collection_date
""").df()

print(result.to_string(index=False))

## 🎤 10 Interview Q&A

### Q1: What's the difference between ROW_NUMBER, RANK, and DENSE_RANK?

> `ROW_NUMBER` always gives unique numbers — someone gets 1 and the other gets 2, even on a tie. `RANK` shares rank on ties but skips numbers after them — so 1, 1, 3. `DENSE_RANK` shares rank on ties but never skips — 1, 1, 2. I use `DENSE_RANK` when I need Top-N per group without gaps in ranking.

---

### Q2: When would you use LAG() vs a self-join?

> `LAG()` is cleaner, more readable, and performs better than a self-join for sequential comparisons. At Citi I used `LAG()` to compare server utilization between collection intervals — a self-join on 6,000 endpoints would have been extremely expensive and the SQL unreadable.

---

### Q3: What's the difference between PARTITION BY and GROUP BY?

> `GROUP BY` collapses rows into one per group — you lose the individual records. `PARTITION BY` keeps all rows and adds the calculation *alongside* them. Window functions never reduce row count.

---

### Q4: How would you calculate a 7-day rolling average?

```sql
AVG(cpu_utilization) OVER (
    PARTITION BY server_id
    ORDER BY collection_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
)
```
> I used this exact pattern at Citi to smooth telemetry noise before feeding data into Prophet capacity forecasting models.

---

### Q5: What is ROWS BETWEEN vs RANGE BETWEEN?

> `ROWS BETWEEN` is physical — it counts actual rows regardless of their values. `RANGE BETWEEN` is logical — it includes all rows with the same `ORDER BY` value as the current row. For time series I always use `ROWS BETWEEN` because it's predictable: if you have duplicate dates, `RANGE` behaves unexpectedly.

---

### Q6: How do you find the top 3 salaries per department?

```sql
WITH ranked AS (
    SELECT *,
        DENSE_RANK() OVER (
            PARTITION BY department
            ORDER BY salary DESC
        ) AS rnk
    FROM employees
)
SELECT * FROM ranked WHERE rnk <= 3
```

---

### Q7: What's a practical use case for LEAD()?

> Detecting state transitions. `LEAD()` lets you peek at the next row's value — useful for finding when a server's status changed from `normal` to `warning`, or identifying gaps in time series data where a collection was missed.

---

### Q8: Can you use window functions in a WHERE clause?

> No — window functions are evaluated **after** `WHERE`. The solution is to wrap them in a CTE or subquery first, then filter on the result. This is a classic interview trap — if you try to filter on a window function directly in `WHERE`, you'll get a syntax error.

---

### Q9: How would you calculate month-over-month revenue growth?

```sql
WITH monthly AS (
    SELECT
        month,
        revenue,
        LAG(revenue) OVER (ORDER BY month) AS prev_month_revenue
    FROM revenue_table
)
SELECT
    month,
    revenue,
    ROUND((revenue - prev_month_revenue) / prev_month_revenue * 100, 1) AS mom_growth_pct
FROM monthly
```

---

### Q10: What's the performance consideration with window functions on large datasets?

> Window functions require sorting and can be expensive at scale. At Citi processing millions of telemetry rows, I always ensured the `PARTITION BY` columns were indexed, and used Athena's columnar Parquet format to minimize data scanned. Also: run `EXPLAIN ANALYZE` first — sometimes a pre-aggregated CTE is cheaper than a broad window.

## 📚 Key Terminology to Master

| Term | Definition | Example |
|------|-----------|----------|
| **Window Function** | Performs calculation across rows related to current row without collapsing | `ROW_NUMBER() OVER (...)` |
| **Window Frame** | The `ROWS/RANGE BETWEEN` clause — defines which rows are included | `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` |
| **Partition** | The `PARTITION BY` clause — defines the group the window operates over | `PARTITION BY server_id` |
| **Ordered Set** | The `ORDER BY` within the window — defines ranking/sequencing | `ORDER BY collection_date` |
| **Rolling Aggregate** | A SUM/AVG computed over a sliding window frame | 7-day rolling average |
| **ROWS vs RANGE** | `ROWS` = physical row count (use this); `RANGE` = logical value equality (avoid for time series) | `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` |
| **Columnar Format** | Parquet/ORC storage — scans only needed columns, critical for window function performance | Athena on Parquet |
| **CTE** | Common Table Expression — required wrapper when filtering on window function results | `WITH ranked AS (...)` |

## 💼 The Citi Narrative: 6,000 Endpoints, Zero Self-Joins

### The Problem

At Citi, I processed telemetry from **6,000+ endpoints** — CPU, memory, network, and application transaction rates collected every 5 minutes. The early SQL code was littered with expensive self-joins to compare current metrics to previous intervals. A single self-join on 6,000 servers across millions of rows took 8-12 minutes. Capacity reports ran overnight.

### The Solution

I rewrote the entire reporting layer using **window functions**:
1. `LAG()` to detect utilization spikes by comparing current vs previous collection interval
2. Rolling `AVG()` with 7-day frames to smooth noise before feeding Prophet forecasting models
3. `DENSE_RANK()` to identify the top-N most loaded servers per region without subqueries

### The Code Pattern

```sql
WITH telemetry AS (
    SELECT
        server_id,
        region,
        collection_time,
        cpu_pct,
        -- Spike detection: no self-join needed
        cpu_pct - LAG(cpu_pct) OVER (PARTITION BY server_id ORDER BY collection_time)
            AS cpu_delta,
        -- Trend smoothing for forecasting
        AVG(cpu_pct) OVER (
            PARTITION BY server_id
            ORDER BY collection_time
            ROWS BETWEEN 2015 PRECEDING AND CURRENT ROW  -- 7 days × 288 intervals/day
        ) AS rolling_7d_avg,
        -- Hot-server ranking per region
        DENSE_RANK() OVER (PARTITION BY region ORDER BY cpu_pct DESC) AS load_rank
    FROM endpoint_telemetry
    WHERE collection_time >= CURRENT_DATE - INTERVAL '30 days'
)
SELECT * FROM telemetry
WHERE cpu_delta > 20      -- alert: spike > 20 points
   OR load_rank <= 5      -- top 5 busiest per region
```

### The Impact

- **Performance:** Report runtime dropped from 8-12 minutes to under 90 seconds
- **Reliability:** Eliminated complex self-join logic that frequently produced duplicate rows
- **Forecasting:** Rolling window output fed directly into Prophet — 6-month ahead capacity forecasts
- **Scale:** Same queries run unchanged on Athena against S3 Parquet at petabyte scale

## 💪 Practice Exercises

In [ ]:
# EXERCISE 1: Find employees whose salary is above their department average
# Use a window function — no subquery or self-join

result = con.execute("""
WITH dept_avg AS (
    SELECT
        name,
        department,
        salary,
        ROUND(AVG(salary) OVER (PARTITION BY department), 0) AS dept_avg_salary
    FROM employees
)
SELECT name, department, salary, dept_avg_salary,
       salary - dept_avg_salary AS above_avg_by
FROM dept_avg
WHERE salary > dept_avg_salary
ORDER BY department, above_avg_by DESC
""").df()

print("Exercise 1 — above-average earners:")
print(result.to_string(index=False))
# Expected: Alice(120k), Bob(120k) in Eng; Eve+Frank(105k) in Mkt; Henry+Ivy(115k) in Fin

In [ ]:
# EXERCISE 2: Find dates where server-01 CPU was trending upward
# (current day higher than previous day AND higher than day before that)

result = con.execute("""
WITH lags AS (
    SELECT
        server_id,
        collection_date,
        cpu_utilization,
        LAG(cpu_utilization, 1) OVER (PARTITION BY server_id ORDER BY collection_date) AS prev_1,
        LAG(cpu_utilization, 2) OVER (PARTITION BY server_id ORDER BY collection_date) AS prev_2
    FROM server_metrics
    WHERE server_id = 'server-01'
)
SELECT collection_date, prev_2, prev_1, cpu_utilization
FROM lags
WHERE cpu_utilization > prev_1 AND prev_1 > prev_2
ORDER BY collection_date
""").df()

print("Exercise 2 — server-01 continuous upward trend:")
print(result.to_string(index=False))
# Expected: rows where CPU is consistently rising over 3 days

In [ ]:
# EXERCISE 3: Calculate the percentile rank of each employee's salary
# within their department (0.0 = lowest, 1.0 = highest)
# Hint: use PERCENT_RANK() — same OVER syntax as other window functions

result = con.execute("""
SELECT
    name,
    department,
    salary,
    ROUND(PERCENT_RANK() OVER (
        PARTITION BY department
        ORDER BY salary
    ), 2) AS salary_percentile
FROM employees
ORDER BY department, salary_percentile DESC
""").df()

print("Exercise 3 — salary percentile rank within department:")
print(result.to_string(index=False))
# Expected: top earners have percentile = 1.0

## 🎯 Summary: Key Takeaways

### What You've Learned

1. ✅ **Window vs GROUP BY:** Window functions keep all rows; `GROUP BY` collapses them
2. ✅ **The Three Rankers:** `ROW_NUMBER` (unique), `RANK` (skips), `DENSE_RANK` (never skips) — use DENSE_RANK for Top-N
3. ✅ **LAG / LEAD:** Time-travel within a partition — no self-join, no subquery
4. ✅ **Rolling Frames:** `ROWS BETWEEN n PRECEDING AND CURRENT ROW` — use `ROWS`, not `RANGE` for time series
5. ✅ **The CTE Rule:** Window functions cannot go in `WHERE` — wrap in CTE first
6. ✅ **Performance:** Index `PARTITION BY` columns; use columnar format (Parquet/Athena) at scale
7. ✅ **PERCENT_RANK / NTILE:** Additional ranking functions for percentile and bucket analysis

### When to Use Window Functions

- ✅ **DO** use `DENSE_RANK` for Top-N per group problems
- ✅ **DO** use `LAG/LEAD` instead of self-joins for sequential comparisons
- ✅ **DO** use rolling `AVG/SUM` for trend smoothing and running totals
- ❌ **DON'T** put window functions in `WHERE` — use a CTE
- ❌ **DON'T** use `RANGE BETWEEN` for time series — use `ROWS BETWEEN`

### Interview Confidence Checklist

- [ ] Can explain the difference between `ROW_NUMBER`, `RANK`, and `DENSE_RANK` precisely
- [ ] Can write the Top-N per group query from scratch
- [ ] Can write a `LAG()` query for month-over-month comparison
- [ ] Can write a 7-day rolling average with correct `ROWS BETWEEN` syntax
- [ ] Can explain why window functions fail in `WHERE` — and the CTE solution
- [ ] Can explain `ROWS BETWEEN` vs `RANGE BETWEEN` with a concrete example
- [ ] Can name Citi's use case: 6,000 endpoints, spike detection, capacity forecasting

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Window functions separate junior from senior SQL writers. Master them and you've mastered the most-asked SQL topic in data engineering interviews. 🚀